# OpenJev

[openjev/openjev](https://huggingface.co/openjev/openjev), served as [openjev-FP8](https://huggingface.co/openjev/openjev-FP8) by `./serve.sh` (start it first, in a terminal). Each question is one forward pass whose answer is read from the option-letter scores. Edit `CASES` to try your own states and questions.

In [4]:
import json
import subprocess
import time

import httpx

URL = "http://127.0.0.1:3000/v1"  # the vendor helper in front of vLLM; serve.sh picks the checkpoint
client = httpx.Client(timeout=120)


def ask(state, questions):
    start = time.perf_counter()
    r = client.post(f"{URL}/systemone", json={"model": "openjev", "state": state, "questions": questions})
    r.raise_for_status()
    return r.json(), (time.perf_counter() - start) * 1000


def vram():
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    return out


In [5]:
print(json.dumps(client.get(f"{URL}/version").json(), indent=1))
print(vram())

{
 "model_dir": "openjev-FP8",
 "T": 0.85,
 "noul_t": 1.829074,
 "noul_bias": 0.0,
 "flags": {
  "perms": 1,
  "stagger": true,
  "loop_break": false,
  "compact": false,
  "compact_cap": 0,
  "layout": "",
  "pad": 0,
  "targeted": true,
  "instr_style": "pyrepr"
 },
 "shim_file": "shim.py",
 "shim_sha256": "81a22f1b1b8912a465059207ef9f60b7c6c16b4de6372305d867efbe38a1987a"
}
NVIDIA GeForce RTX 5090, 29468 MiB, 32607 MiB


In [7]:
# (state, questions). Types: choice (criteria = {option: description or None}), noul (yes/no), score (ordered levels).
CASES = [
    (
        "Customer message: I was charged twice for my order last week and nobody has replied.",
        {
            "route": {"type": "choice", "instructions": "Which team should handle this?",
                      "criteria": {"billing": "charges, refunds, invoices", "shipping": "delivery and tracking",
                                   "technical": "bugs and login problems"}},
            "angry": {"type": "noul", "instructions": "Is the customer angry?"},
            "urgency": {"type": "score", "instructions": "How urgent is this?",
                        "criteria": ["can wait", "this week", "today", "right now"]},
        },
    ),
    (
        # Grounding judge with a subtle mismatch: the answer gets the year right but the amount wrong.
        "Source: The company reported revenue of $4.2 billion for fiscal 2025, up 8% from the prior year.\n"
        "Answer: Revenue in fiscal 2025 was $4.8 billion, an 8% increase.",
        {
            "grounded": {"type": "noul", "instructions": "Is every claim in the answer supported by the source?"},
            "verdict": {"type": "choice", "instructions": "How does the answer relate to the source?",
                        "criteria": {"faithful": None, "contains a wrong number": None,
                                     "adds unsupported claims": None, "off topic": None}},
        },
    ),
    (
        "Post: Selling my old account with 3 years of history, DM me, payment in gift cards only.",
        {
            "policy": {"type": "choice", "instructions": "Which policy does this post violate, if any?",
                       "criteria": {"none": None, "spam": "unsolicited advertising", "account trading": "buying or selling accounts",
                                    "harassment": None, "scam": "likely fraud"}},
        },
    ),
    (
        {"goal": "Change the shipping address on order #1182",
         "page": {"url": "https://shop.example/orders/1182", "title": "Order #1182"},
         "elements": [{"index": 0, "tag": "a", "text": "Track package"},
                      {"index": 1, "tag": "button", "text": "Cancel order"},
                      {"index": 2, "tag": "button", "text": "Edit delivery details"},
                      {"index": 3, "tag": "a", "text": "Contact support"},
                      {"index": 4, "tag": "button", "text": "Buy it again"}]},
        {
            "element": {"type": "choice", "instructions": "Which element should the agent click next to reach the goal?",
                        "criteria": {"0": "Track package", "1": "Cancel order", "2": "Edit delivery details",
                                     "3": "Contact support", "4": "Buy it again"}},
            "done": {"type": "noul", "instructions": "Is the goal already achieved on this page?"},
        },
    ),
    # The cases from GLiNER2.5-Decide/try.py, as OpenJev questions, to compare the two on the same inputs.
    (
        "Battery dies before lunch, but the keyboard and the screen are the best I have used on a laptop.",
        {"sentiment": {"type": "choice", "instructions": "What is the sentiment of this review?",
                       "criteria": {"positive": None, "negative": None, "mixed": None, "neutral": None}}},
    ),
    (
        "Hi, I was charged twice for my March invoice and I'd like the duplicate refunded.",
        {
            "intent": {"type": "choice", "instructions": "What does the customer want?",
                       "criteria": {"refund_request": "wants money back",
                                    "billing_question": "asks about a charge without asking for money back",
                                    "cancel_subscription": "wants to stop the service",
                                    "technical_issue": "something is broken"}},
            "urgency": {"type": "score", "instructions": "How urgent is this message?", "criteria": ["low", "medium", "high"]},
        },
    ),
    (
        # GLiNER runs this as one multi-label task; OpenJev has no multi-label type, so it's one yes/no per topic.
        "The deploy failed because the database migration timed out; on-call rolled back and paged the DBA.",
        {f"topic_{t}": {"type": "noul", "instructions": f"Is this text about {t}?"}
         for t in ["deployment", "database", "incident", "billing", "hiring"]},
    ),
    (
        "Policy: refunds are allowed within 30 days of purchase. The customer bought the item 45 days ago.",
        {"refund_allowed": {"type": "choice", "instructions": "Is the customer eligible for a refund under the policy?",
                            "criteria": {"yes": None, "no": None, "not enough information": None}}},
    ),
]

In [8]:
ask(*CASES[0])  # warm-up, so the timings below are steady-state

for state, questions in CASES:
    result, ms = ask(state, questions)
    print(f"\n=============================\n{state if isinstance(state, str) else state['goal']}\n  ({ms:.0f} ms, {result['usage']['input_tokens']} prompt tokens)")
    for key, a in result["answers"].items():
        shown = {k: v for k, v in a.items() if k not in ("type", "legend")}
        print(f"  {key}: {json.dumps(shown)}")


Customer message: I was charged twice for my order last week and nobody has replied.
  (856 ms, 251 prompt tokens)
  route: {"choice": "billing", "probabilities": {"billing": 0.9999, "shipping": 0.0, "technical": 0.0}, "confidence": 0.9999}
  angry: {"noul": 0.5401}
  urgency: {"score": 2.0367, "probabilities": {"0": 0.0025, "1": 0.1219, "2": 0.712, "3": 0.1636}, "confidence": 0.7095}

Source: The company reported revenue of $4.2 billion for fiscal 2025, up 8% from the prior year.
Answer: Revenue in fiscal 2025 was $4.8 billion, an 8% increase.
  (691 ms, 236 prompt tokens)
  grounded: {"noul": 0.0357}
  verdict: {"choice": "contains a wrong number", "probabilities": {"faithful": 0.0037, "contains a wrong number": 0.9933, "adds unsupported claims": 0.0028, "off topic": 0.0003}, "confidence": 0.991}

Post: Selling my old account with 3 years of history, DM me, payment in gift cards only.
  (330 ms, 99 prompt tokens)
  policy: {"choice": "account trading", "probabilities": {"none": 0.00

## Medical document types

The `--docs` task from GLiNER2.5-Decide/try.py: put .md files or folders in `DOCS` and each file becomes the state of one 15-way choice. OpenJev reads up to 16,384 prompt tokens, so unlike GLiNER there's no chunking. A longer doc fails with an error.

In [11]:
from pathlib import Path

DOCS = [Path("/mnt/c/Users/ClaudeCowork/Dropbox/HealthWiki/Paul/documents/")]  # e.g. [Path("~/notes").expanduser(), Path("a.md")]; folders are searched recursively

# Descriptions steer the model, so edit them before adding labels.
DOC_TYPE = {
    "type": "choice",
    "instructions": "What type of medical document is this?",
    "criteria": {
        "discharge_summary": "summary written when a patient leaves hospital: stay, treatment, follow-up",
        "history_and_physical": "admission or first-visit note with history, exam, assessment and plan",
        "progress_note": "routine note on an ongoing patient, often SOAP format",
        "consultation_note": "specialist's opinion requested by another clinician",
        "operative_report": "description of a surgery or procedure as performed",
        "radiology_report": "findings and impression from imaging such as X-ray, CT, MRI or ultrasound",
        "pathology_report": "microscopic or gross findings on a tissue or biopsy specimen",
        "lab_report": "laboratory test results with values and reference ranges",
        "prescription": "medication order with drug, dose, route and frequency",
        "referral_letter": "letter asking another clinician or service to see a patient",
        "patient_education": "plain-language information or instructions written for patients",
        "clinical_guideline": "recommendations or protocol for managing a condition",
        "research_article": "study with methods, results and discussion",
        "billing_or_insurance": "claims, codes, prior authorization or payment",
        "other": "not a medical document, or none of the above",
    },
}

files = sorted({f for p in DOCS for f in (p.rglob("*.md") if p.is_dir() else [p])})
if not files:
    print("DOCS is empty: add .md files or folders above")
for f in files:
    result, ms = ask(f.read_text(), {"doc_type": DOC_TYPE})
    a = result["answers"]["doc_type"]
    top = sorted(a["probabilities"].items(), key=lambda kv: -kv[1])[:3]
    print(f"\n=============================\n{f}\n  {a['choice']} ({a['confidence']:.2f})  top3 {top}  ({ms:.0f} ms, {result['usage']['input_tokens']} tokens)")


/mnt/c/Users/ClaudeCowork/Dropbox/HealthWiki/Paul/documents/2006/2006-07-04_other_unknown_e31394.md
  other (0.76)  top3 [('other', 0.7734), ('discharge_summary', 0.1324), ('history_and_physical', 0.0735)]  (10052 ms, 1630 tokens)

/mnt/c/Users/ClaudeCowork/Dropbox/HealthWiki/Paul/documents/2006/2006-11-14_vaccination-record_unknown_c145e9.md
  other (1.00)  top3 [('other', 0.9954), ('prescription', 0.0022), ('patient_education', 0.0006)]  (2055 ms, 1028 tokens)

/mnt/c/Users/ClaudeCowork/Dropbox/HealthWiki/Paul/documents/2006/2006-11_other_detskaya-poliklinika-4_442d6f.md
  other (0.57)  top3 [('other', 0.5955), ('discharge_summary', 0.2127), ('history_and_physical', 0.1836)]  (2605 ms, 1490 tokens)

/mnt/c/Users/ClaudeCowork/Dropbox/HealthWiki/Paul/documents/2016/2016-04-15_study-report_unknown_8e95d0.md
  other (0.76)  top3 [('other', 0.7802), ('radiology_report', 0.1336), ('lab_report', 0.0742)]  (1996 ms, 1196 tokens)

/mnt/c/Users/ClaudeCowork/Dropbox/HealthWiki/Paul/documents/2

RemoteProtocolError: Server disconnected without sending a response.